# Imports 

In [1]:
import pandas as pd
from collections import defaultdict
import sys
import os
import shutil as sh
import urllib
import tarfile
import numpy as np
import math
import seaborn as sns
import glob, os
import importlib
import gzip
import MDAnalysis as mda
import nglview as nv
import requests
import json
from biopandas.pdb import PandasPdb
from Bio import AlignIO
import re
from io import StringIO

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.spatial import ConvexHull

from urllib.error import HTTPError
from pathlib import Path
from ipywidgets import interact, interactive, fixed, interact_manual, IntProgress
import ipywidgets as widgets # type: ignore
from IPython.display import display, Markdown, clear_output

#Pandarallel works only on linux and mac
try:
    from pandarallel import pandarallel
    pandarallel.initialize(nb_workers=8,progress_bar=True)
    PARRALEL = True
except:
    PARRALEL = False

from tqdm.notebook import tnrange, tqdm
tqdm.pandas() #activate tqdm progressbar for pandas apply

#Pandas configuration
pd.options.mode.chained_assignment = (
    None  # default='warn', remove pandas warning when adding a new column
)

pd.set_option("display.max_columns", None)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
%config InlineBackend.figure_format ='svg' #better quality figure figure

#%matplotlib inline
sns.set_style("darkgrid")

np.seterr(divide='ignore', invalid='ignore')



INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

# Methods

In [19]:
from sklearn.calibration import LabelEncoder
from sklearn.discriminant_analysis import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


def load_data(path):
    """
    Load data from the path were it was stored.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"The file at {path} was not found.")
    return pd.read_csv(path)

def preprocess_data(df, columns_to_drop=[], categorical_cols=[]):
    """
    Drop uneccessary columns, 
    Encode categorical features, 
    Handle missing values
    """
    encodings: dict[str, np.ndarray] = {}

    # Drop unwanted columns
    if columns_to_drop:
        drop_list = [c for c in columns_to_drop if c in df.columns]
        df = df.drop(columns=drop_list)

    # Label‐encode only the columns listed in categorical_cols
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col].astype(str))
                encodings[col] = le.classes_.copy()
                

    # Identify numeric columns and build the pipeline
    num_list = df.select_dtypes(include=[np.number]).columns.tolist()

    num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean'))])
    df[num_list] = num_pipeline.fit_transform(df[num_list])

    return df, encodings

## Download complete dataset with all anotation at residue-level

In [5]:
import zipfile
import os

zip_path = "/home/user_stel/AISB/Project/Ressources/datasets/S2 File.csv.zip"
# Remove the “.zip” to get the output directory name
extract_dir = zip_path[:-4]

# Make sure the target folder exists (or create it)
os.makedirs(extract_dir, exist_ok=True)

try:
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
except zipfile.BadZipFile:
    print("⚠️ Warning: BadZipFile raised – the .zip may be corrupted.")
except Exception as e:
    print(f"⚠️ Warning: {e}")

print("✅ Extracted all files into:", extract_dir)


✅ Extracted all files into: /home/user_stel/AISB/Project/Ressources/datasets/S2 File.csv


In [6]:

path_to_data = "/home/user_stel/AISB/Project/Ressources/datasets/S2 File.csv/S2 File.csv"

df = load_data(path_to_data)

print(df)

       domain  cathpdb   pdb uniprot_acc  uniprot_id residue_name    IBS  \
0          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
1          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
2          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
3          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
4          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
...       ...      ...   ...         ...         ...          ...    ...   
183885    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS  False   
183886    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS  False   
183887    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS  False   
183888    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS  False   
183889    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS   True   

       chain_id  residue_number  b_factor sec_struc sec_struc_full prot_block  \
0     

In [ ]:
# get column names 
print("Column names:")
for col in df.columns:
    print(col)

Column names:
domain
cathpdb
pdb
uniprot_acc
uniprot_id
residue_name
IBS
chain_id
residue_number
b_factor
sec_struc
sec_struc_full
prot_block
data_type
Experimental Method
resolution
RSA_total_freesasa_tien
convhull_vertex
protrusion
is_hydrophobic_protrusion
is_co_insertable
neighboursList
density
exposed
S35
S60
S95
S100
uniref50
uniref90
uniref100
origin
location
taxon


In [21]:
# get unique domain entries 
unique_domains = df["domain"].unique()
print(unique_domains)


['PH' 'C2' 'START' 'C1' 'C2DIS' 'PX' 'PLD' 'ANNEXIN' 'PLA']


## Data Exploration 

In [7]:
# Temporary fix: redefine type to have HIS as polar.
AATYPE = {
    "LEU": "Hydrophobic,H-non-aromatic",
    "ILE": "Hydrophobic,H-non-aromatic",
    "CYS": "Hydrophobic,H-non-aromatic",
    "MET": "Hydrophobic,H-non-aromatic",
    "TYR": "Hydrophobic,H-aromatic",
    "TRP": "Hydrophobic,H-aromatic",
    "PHE": "Hydrophobic,H-aromatic",
    "HIS": "Polar",
    "LYS": "Positive",
    "ARG": "Positive",
    "ASP": "Negative",
    "GLU": "Negative",
    "VAL": "Non-polar",
    "ALA": "Non-polar",
    "SER": "Polar",
    "ASN": "Polar",
    "GLY": "Non-polar",
    "PRO": "Non-polar",
    "GLN": "Polar",
    "THR": "Polar",
    "UNK": "none"
}
df["type"] = df.residue_name.apply(lambda x: AATYPE[x])
print(df)

       domain  cathpdb   pdb uniprot_acc  uniprot_id residue_name    IBS  \
0          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
1          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
2          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
3          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
4          PH  2m14A00  2M14      P32776  TFB1_YEAST          ASN  False   
...       ...      ...   ...         ...         ...          ...    ...   
183885    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS  False   
183886    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS  False   
183887    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS  False   
183888    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS  False   
183889    PLA  5p2pA00  5P2P      P00592   PA21B_PIG          CYS   True   

       chain_id  residue_number  b_factor sec_struc sec_struc_full prot_block  \
0     

1. **Target column:   IBS**
    false(non-IBS)             → 0
    true(IBS)                  → 1
2. Feature column:  **convhull_vertex**
    false                      → 0
    true                       → 1
3. Feature column:  **protrusion**
    false                      → 0
    true                       → 1
4. Feature column:  **is_hydrophobic_protrusion**
    false                      → 0
    true                       → 1
5. Feature column:  **is_co_insertable**
    false                      → 0
    true                       → 1
6. Feature column:  **exposed**
    false                      → 0
    true                       → 1
7. Feature column:  **type**
    Hydrophobic,H-aromatic     → 0
    Hydrophobic,H-non-aromatic → 1
    Negative                   → 2
    Non-polar                  → 3
    Polar                      → 4
    Positive                   → 5


In [21]:
cols_to_drop = ["cathpdb", "pdb", "chain_id", "uniprot_acc", "uniprot_id", "b_factor", "sec_struc", "sec_struc_full", "prot_block", "data_type", 
                "Experimental Method", "resolution", "RSA_total_freesasa_tien", "neighboursList", "density", "S35", "S60", "S95",
                "S100", "uniref50", "uniref90", "uniref100", "origin", "location", "taxon"]
categorical_cols = ["convhull_vertex", "protrusion", "is_hydrophobic_protrusion", "is_co_insertable", "exposed", "type"]
df_processed, encodings = preprocess_data(df, columns_to_drop=cols_to_drop, categorical_cols=categorical_cols)
print(df_processed)

print("Mapping for 'type':")
for code, label in enumerate(encodings["type"]):
    print(f"  {code} → {label}")

       domain residue_name    IBS  residue_number  convhull_vertex  \
0          PH          ASN  False            19.0              0.0   
1          PH          ASN  False            75.0              1.0   
2          PH          ASN  False            78.0              1.0   
3          PH          ASN  False            92.0              1.0   
4          PH          ASN  False            93.0              1.0   
...       ...          ...    ...             ...              ...   
183885    PLA          CYS  False            91.0              0.0   
183886    PLA          CYS  False            96.0              0.0   
183887    PLA          CYS  False            98.0              0.0   
183888    PLA          CYS  False           105.0              0.0   
183889    PLA          CYS   True           124.0              0.0   

        protrusion  is_hydrophobic_protrusion  is_co_insertable  exposed  type  
0              0.0                        0.0               0.0      1.0   4.0

In [20]:
# Count how many residues are labeled as “IBS” vs “non‐IBS”:
df["IBS"].value_counts()

# How many hydrophobic protrusions reside in IBS vs non‐IBS?
df[df["is_hydrophobic_protrusion"] & (df["IBS"] == 1)].shape[0]


IBS
False    156007
True      27883
Name: count, dtype: int64

1454